In [1]:
import numpy as np
from space_exploration.beans.dataset_bean import Dataset
from space_exploration.dataset.transforms.AllTransforms import TransformationReferences
dataset_name = "re200-sr1etot"
dataset = Dataset.get_dataset_or_fail(dataset_name)
# ds = dataset.get_training_dataset(64, TransformationReferences.DEFAULT_UNCHANGED.transformation, TransformationReferences.DEFAULT_UNCHANGED.transformation, size=1000)
ds = dataset.get_training_dataset(64, TransformationReferences.COMPONENT_NORMALIZE.transformation, TransformationReferences.Y_ALONG_COMPONENT_NORMALIZE.transformation, size=1000)
xs = np.array([x.numpy() for x, y in ds])
ys = np.array([y.numpy() for x, y in ds])

Loading std & mean of dataset re200-sr1etot
Loading stds & means of dataset re200-sr1etot
⌛ Initializing Dataset...
X...
[########################################] | 100% Completed | 820.23 ms
Y...
[########################################] | 100% Completed | 8.58 ss


In [2]:
import torch
import torch.nn.functional as F

def get_gaussian_kernel2d(kernel_size=5, sigma=1.0, device='cpu'):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=device) - kernel_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g /= g.sum()
    g2d = g[:, None] * g[None, :]  # outer product for 2D kernel
    g2d = g2d.unsqueeze(0).unsqueeze(0)  # shape (1,1,k,k)
    return g2d

def apply_dog_2d(images, sigma1=1.0, sigma2=2.0, kernel_size=9):
    """
    ys: torch.Tensor of shape (B, C, H, W)
    returns: DoG-filtered tensor, same shape
    """
    B, C, H, W = images.shape
    device = images.device
    
    # Get both Gaussian kernels
    g1 = get_gaussian_kernel2d(kernel_size, sigma1, device=device)
    g2 = get_gaussian_kernel2d(kernel_size, sigma2, device=device)
    
    # Depthwise convolution for both sigmas
    blurred1 = F.conv2d(images, g1, padding="same")
    blurred2 = F.conv2d(images, g2, padding="same")

    # Difference of Gaussians
    return blurred1 - blurred2


In [3]:
import numpy as np
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter

# Arbitrary value, more should be overkill
max_sigma = 30

from pathlib import Path

search_folder = Path("dog_search_abs")
search_folder.mkdir(parents=True, exist_ok=True)

import pickle
import pandas as pd

import torch

import ipywidgets as widgets
from IPython.display import display
import time
import random


def search_at_layer(y_layer, component_in, component_out):

    out_file = search_folder / f"{y_layer}-{component_in}-{component_out}.pkl"

    log_data = dict()

    nb_iter = 100

    task_id = f"L{y_layer}-IC{component_in}-OC{component_out}"

    def get_base_correlation(x, y):
        correlations = [pearsonr(x[i].ravel(), y[i].ravel())[0] for i in range(x.shape[0])]
        return np.mean(correlations)
    
    def get_correlation(x, y, s1, s2):
        # x: B, 1, 64, 64
        # y : B, 1, 64, 64
        x = torch.tensor(x, device="cuda")
        dog = apply_dog_2d(x, sigma1=s1, sigma2=s2, kernel_size=63).cpu()
        correlations = [pearsonr(dog[i].ravel(), y[i].ravel())[0] for i in range(dog.shape[0])]
        return np.mean(correlations)

    # xs shape: B, C, 64, 1, 64 --> images shape: B, 1, 64, 64
    images = xs[:, component_in: component_in + 1, :, 0, :]
    # ys shape: B, C, 64, 64, 64 --> fields shape: B, 1, 64, 64
    fields = ys[:, component_out: component_out + 1, :, y_layer, :]

    base_corr = abs(get_base_correlation(images, fields))
    log_data["baseline"] = base_corr

    def get_fit_score(s1, s2):
        average_y_correlations = abs(get_correlation(images, fields, s1, s2))
        if np.isnan(average_y_correlations).any():
            print("WARNING NAN VALUE")
            return -100000
        return average_y_correlations - base_corr
        
    import optuna
        
    search_data = []
    try:

        def objective(trial):
            s1 = trial.suggest_float("s1", 0.1, 29.0)  # max 29 to allow offset up to 1.0
            s2 = s1 + trial.suggest_float("offset", 1.0, 30.0 - s1)
            return get_fit_score(s1, s2)

        study = optuna.create_study(direction="maximize", study_name=task_id)
        study.optimize(objective, n_trials=nb_iter, n_jobs=60)

    except KeyboardInterrupt as e:
        pass

    search_data = [
        {
            "trial": trial.number,
            "s1": trial.params.get("s1"),
            "s2": trial.params.get("offset") + trial.params.get("s1"),
            "score": trial.value
        }
        for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE
    ]

    search_df = pd.DataFrame(search_data)
    log_data["search_df"] = search_df

    with open(out_file, "wb") as file:
        pickle.dump(log_data, file)

    return search_df

In [ ]:
for comp_in in range(3):
    for comp_out in range(3):
        for layer in range(1, 64):
            search_at_layer(layer, comp_in, comp_out)

[I 2025-07-28 14:05:37,129] A new study created in memory with name: L1-IC0-OC0


In [ ]:
loaded_data